# Kopi IoT Web3 — Deep Learning Forecasting (Keras) — Multi-step

Mengajarkan **ANN → DNN → LSTM** + membandingkannya dengan **Naive, Linear, Prophet**
untuk **3 variabel** (suhu, udara, tanah).

**Evaluasi SETARA (multi-step / iteratif):** semua model memprediksi K hari ke depan
dari titik potong **tanpa melihat data aktual** (model windowed memakai prediksinya sendiri
sebagai input berikutnya — recursive). Ini adil & sesuai pemakaian nyata di website.

> Data masih sedikit (~60 hari) → DL belum tentu menang; itu pelajaran (overfitting & data hunger).
> Kurva Loss/MAE per epoch di sini OTENTIK — keunggulan mengajar pakai neural network.

Jalankan: Runtime -> Run all.

In [ ]:
!pip -q install prophet

## 1. Ambil & bersihkan data

In [ ]:
import requests, pandas as pd, numpy as np

CHANNEL_ID = 1848413
url = f'https://api.thingspeak.com/channels/{CHANNEL_ID}/feeds.json'
feeds = requests.get(url, params={'results':8000,'timezone':'Asia/Jakarta'}, timeout=30).json()['feeds']

df = pd.DataFrame(feeds)
df['created_at'] = pd.to_datetime(df['created_at'])
if df['created_at'].dt.tz is not None: df['created_at'] = df['created_at'].dt.tz_localize(None)
for col,name in [('field1','suhu'),('field2','udara'),('field3','tanah')]:
    df[name] = pd.to_numeric(df[col], errors='coerce')
df = df[['created_at','suhu','udara','tanah']].set_index('created_at')

RANGES = {'suhu':(10,45),'udara':(0,100),'tanah':(0,100)}
for v,(lo,hi) in RANGES.items(): df[v] = df[v].where((df[v]>=lo)&(df[v]<=hi))
for v in RANGES:
    s=df[v]; med=s.median(); mad=(s-med).abs().median()
    if mad and mad>0: df[v]=s.where((s-med).abs()<=3.5*1.4826*mad)

harian = df.resample('D').mean().interpolate(method='time', limit=3, limit_area='inside').dropna(how='all')
print('Jumlah hari:', len(harian)); harian.tail()

## 2. Metrik & model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.linear_model import LinearRegression
from prophet import Prophet
import logging
logging.getLogger('prophet').setLevel(logging.WARNING); logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
tf.random.set_seed(42); np.random.seed(42)

def mae(a,p):  a,p=np.asarray(a,float),np.asarray(p,float); return float(np.mean(np.abs(a-p)))
def rmse(a,p): a,p=np.asarray(a,float),np.asarray(p,float); return float(np.sqrt(np.mean((a-p)**2)))
def mape(a,p):
    a,p=np.asarray(a,float),np.asarray(p,float); m=np.abs(a)>1e-6
    return float(np.mean(np.abs((a[m]-p[m])/a[m]))*100) if m.sum() else float('nan')
def akurasi(a,p): return float(max(0.0, 100-mape(a,p)))

def build_mlp(W):  m=models.Sequential([layers.Input((W,)), layers.Dense(16,activation='relu'), layers.Dense(1)]); m.compile(optimizer='adam', loss='mse', metrics=['mae']); return m
def build_dnn(W):  m=models.Sequential([layers.Input((W,)), layers.Dense(32,activation='relu'), layers.Dropout(0.2), layers.Dense(16,activation='relu'), layers.Dense(1)]); m.compile(optimizer='adam', loss='mse', metrics=['mae']); return m
def build_lstm(W): m=models.Sequential([layers.Input((W,1)), layers.LSTM(16), layers.Dense(1)]); m.compile(optimizer='adam', loss='mse', metrics=['mae']); return m

## 3. Fungsi analisis (multi-step untuk semua model)

In [ ]:
def analisis(VAR, W=7, K=10, epochs=400):
    s = harian[VAR].dropna(); dates = list(s.index); vals = s.values.astype('float32')
    train_vals, test_actual, test_dates = vals[:-K], vals[-K:], dates[-K:]

    # windowing dari TRAIN saja
    X,y = [],[]
    for i in range(len(train_vals)-W): X.append(train_vals[i:i+W]); y.append(train_vals[i+W])
    X,y = np.array(X), np.array(y)
    mn,mx = float(X.min()), float(X.max())
    norm   = lambda a:(a-mn)/(mx-mn+1e-9)
    denorm = lambda a:a*(mx-mn+1e-9)+mn

    es = callbacks.EarlyStopping(patience=40, restore_best_weights=True)
    fit = lambda m,xx: m.fit(xx, norm(y), validation_split=0.2, epochs=epochs, batch_size=8, verbose=0, callbacks=[es])
    mlp=build_mlp(W);  h_mlp =fit(mlp,  norm(X))
    dnn=build_dnn(W);  h_dnn =fit(dnn,  norm(X))
    lstm=build_lstm(W);h_lstm=fit(lstm, norm(X).reshape(-1,W,1))
    lr = LinearRegression().fit(X, y)

    # Prophet: dilatih pada data sebelum tanggal uji (multi-step alami)
    dfp = harian[[VAR]].dropna().reset_index(); dfp.columns=['ds','y']
    if dfp['ds'].dt.tz is not None: dfp['ds']=dfp['ds'].dt.tz_localize(None)
    mP = Prophet(weekly_seasonality=False, daily_seasonality=False, yearly_seasonality=False, changepoint_prior_scale=0.01)
    mP.fit(dfp[dfp['ds'] < test_dates[0]])
    p_prophet = mP.predict(pd.DataFrame({'ds':test_dates}))['yhat'].values

    # Recursive multi-step untuk model windowed
    def recursive(step, last_window):
        win = last_window.astype('float32').copy(); out=[]
        for _ in range(K): yh=step(win); out.append(yh); win=np.append(win[1:], yh)
        return np.array(out)
    lw = train_vals[-W:]
    preds = {
        'Naive':     np.repeat(float(train_vals[-1]), K),
        'Linear':    recursive(lambda w: float(lr.predict(w.reshape(1,-1))[0]), lw),
        'Prophet':   p_prophet,
        'MLP (ANN)': recursive(lambda w: float(denorm(mlp.predict(norm(w).reshape(1,-1),  verbose=0)[0,0])), lw),
        'DNN':       recursive(lambda w: float(denorm(dnn.predict(norm(w).reshape(1,-1),  verbose=0)[0,0])), lw),
        'LSTM':      recursive(lambda w: float(denorm(lstm.predict(norm(w).reshape(1,W,1),verbose=0)[0,0])), lw),
    }
    rows = [{'model':k,'MAE':round(mae(test_actual,v),3),'RMSE':round(rmse(test_actual,v),3),'Akurasi(%)':round(akurasi(test_actual,v),1)} for k,v in preds.items()]
    tabel = pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True)
    return {'VAR':VAR,'tabel':tabel,'hist':{'MLP (ANN)':h_mlp,'DNN':h_dnn,'LSTM':h_lstm},'preds':preds,'actual':test_actual,'dates':test_dates}

## 4. Fungsi grafik

In [ ]:
import matplotlib.pyplot as plt

def plot_loss(res):
    fig,ax = plt.subplots(3,2,figsize=(12,11))
    for row,nama in enumerate(['MLP (ANN)','DNN','LSTM']):
        h=res['hist'][nama]
        ax[row,0].plot(h.history['loss'],label='train'); ax[row,0].plot(h.history['val_loss'],label='val')
        ax[row,0].set_title(nama+' - Loss (MSE) - '+res['VAR']); ax[row,0].set_xlabel('epoch'); ax[row,0].grid(alpha=.3); ax[row,0].legend()
        ax[row,1].plot(h.history['mae'],label='train'); ax[row,1].plot(h.history['val_mae'],label='val')
        ax[row,1].set_title(nama+' - MAE'); ax[row,1].set_xlabel('epoch'); ax[row,1].grid(alpha=.3); ax[row,1].legend()
    plt.tight_layout(); plt.show()

def plot_compare(res):
    tt=res['tabel'].set_index('model'); fig,(a1,a2)=plt.subplots(1,2,figsize=(13,4))
    tt[['MAE','RMSE']].plot(kind='bar',ax=a1,color=['#ef4444','#9ca3af']); a1.set_title('Error - '+res['VAR']); a1.grid(alpha=.3,axis='y'); a1.tick_params(axis='x',rotation=30)
    tt['Akurasi(%)'].plot(kind='bar',ax=a2,color='#16a34a'); a2.set_title('Akurasi - '+res['VAR']); a2.set_ylim(0,100); a2.grid(alpha=.3,axis='y'); a2.tick_params(axis='x',rotation=30)
    plt.tight_layout(); plt.show()

def plot_actual(res):
    cmap={'Naive':'#9ca3af','Linear':'#a78bfa','Prophet':'#16a34a','MLP (ANN)':'#ef4444','DNN':'#f59e0b','LSTM':'#3b82f6'}
    plt.figure(figsize=(11,4)); plt.plot(res['dates'],res['actual'],'o-',color='#111',label='Aktual',linewidth=2)
    for nama,p in res['preds'].items(): plt.plot(res['dates'],p,'--',color=cmap.get(nama),label=nama,alpha=.85)
    plt.title('Aktual vs Prediksi (multi-step) - '+res['VAR']); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 5. Variabel: SUHU (dengan kurva loss)

In [ ]:
res_suhu = analisis('suhu')
print('=== SUHU ==='); print(res_suhu['tabel'].to_string(index=False))
plot_loss(res_suhu)

In [ ]:
plot_compare(res_suhu); plot_actual(res_suhu)

## 6. Variabel: UDARA

In [ ]:
res_udara = analisis('udara')
print('=== UDARA ==='); print(res_udara['tabel'].to_string(index=False))
plot_compare(res_udara); plot_actual(res_udara)

## 7. Variabel: TANAH

In [ ]:
res_tanah = analisis('tanah')
print('=== TANAH ==='); print(res_tanah['tabel'].to_string(index=False))
plot_compare(res_tanah); plot_actual(res_tanah)

## 8. Ringkasan gabungan (semua variabel)

In [ ]:
gab = pd.concat([r['tabel'].assign(variabel=r['VAR']) for r in [res_suhu,res_udara,res_tanah]])
print('=== MAE per model x variabel ==='); print(gab.pivot(index='model',columns='variabel',values='MAE').round(3).to_string())
print(); print('=== Akurasi(%) per model x variabel ==='); print(gab.pivot(index='model',columns='variabel',values='Akurasi(%)').round(1).to_string())
gab.pivot(index='model',columns='variabel',values='MAE').plot(kind='bar',figsize=(10,4)); plt.title('MAE per model (multi-step)'); plt.ylabel('MAE'); plt.grid(alpha=.3,axis='y'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 9. Kesimpulan

- Perbandingan kini **setara** (semua multi-step) & realistis (seperti pemakaian website).
- Lihat ringkasan gabungan: **pemenang bisa berbeda tiap variabel** — itu wajar & menarik untuk dibahas.
- Suhu cenderung "lengket" (Naive kuat); udara/tanah lebih fluktuatif (model belajar bisa lebih kompetitif).
- Kurva Loss/MAE = bukti proses belajar neural network (+ deteksi overfitting bila val naik).
- Tangga ajar: Naive -> Linear -> Prophet -> ANN -> DNN -> LSTM.
- Untuk laporan: sajikan MAE, RMSE, DAN Akurasi; tegaskan metode evaluasi (multi-step) & ukuran data.